In [1]:
import pandas as pd
import re

def parse_theorem_file(file_path):
    with open(file_path, 'r') as f:
        content = f.read()
    
    theorem_blocks = re.split(r'\n(?=THEOREM:)', content.strip())
    
    theorems_data = []
    
    for block in theorem_blocks:
        if not block.strip():
            continue
            
        lines = block.strip().split('\n')
        
        theorem_name = None
        goal = None
        expansions = None
        goals = []
        steps = []
        
        for line in lines:
            line = line.strip()
            
            if line.startswith('THEOREM:'):
                theorem_name = line.replace('THEOREM:', '').strip()
            elif line.startswith('GOAL:') and goal is None:
                goal = line.replace('GOAL:', '').strip()
            elif line.startswith('EXPANSIONS:'):
                expansions = int(line.replace('EXPANSIONS:', '').strip())
            elif line.startswith('GOAL:') and goal is not None:
                goals.append(line.replace('GOAL:', '').strip())
            elif line.startswith('STEP:'):
                steps.append(line.replace('STEP:', '').strip())
        
        theorems_data.append({
            'theorem_name': theorem_name,
            'expansions': expansions,
            'goal': goal,
            'sub_goals': goals,
            'steps': steps
        })
    
    return pd.DataFrame(theorems_data)



In [2]:
dfs_seed_sets = (
    {
        "nb": parse_theorem_file('./baseline/longest_theorems_proved_seed1_concat.txt'),
        "nm": parse_theorem_file('./modular/longest_theorems_proved_seed1_concat.txt'),
        "rb": parse_theorem_file('./residual/baseline/longest_theorems_proved_seed1_concat.txt'),
        "rm": parse_theorem_file('./residual/modular/longest_theorems_proved_seed1_concat.txt'),
    },
    {
        "nb": parse_theorem_file('./baseline/longest_theorems_proved_seed2_concat.txt'),
        "nm": parse_theorem_file('./modular/longest_theorems_proved_seed2_concat.txt'),
        "rb": parse_theorem_file('./residual/baseline/longest_theorems_proved_seed2_concat.txt'),
        "rm": parse_theorem_file('./residual/modular/longest_theorems_proved_seed2_concat.txt'),
    },
    {
        "nb": parse_theorem_file('./baseline/longest_theorems_proved_seed3_concat.txt'),
        "nm": parse_theorem_file('./modular/longest_theorems_proved_seed3_concat.txt'),
        "rb": parse_theorem_file('./residual/baseline/longest_theorems_proved_seed3_concat.txt'),
        "rm": parse_theorem_file('./residual/modular/longest_theorems_proved_seed3_concat.txt'),
    },
)

dfs_models_sets = {
    "Normal Baseline":   {
        "prev0": parse_theorem_file("./baseline/longest_theorems_proved_old.txt"),
        **{f"seed{s+1}": dfs_seed_sets[s]["nb"] for s in range(3)},
    },
    "Normal Modular":   {
        "prev0": parse_theorem_file("./modular/longest_theorems_proved_old.txt"),
        **{f"seed{s+1}": dfs_seed_sets[s]["nm"] for s in range(3)},
    },
    "Residual Baseline":   {
        "prev0": parse_theorem_file("./residual/baseline/longest_theorems_proved_old.txt"),
        **{f"seed{s+1}": dfs_seed_sets[s]["rb"] for s in range(3)},
    },
    "Residual Modular":   {
        "prev0": parse_theorem_file("./residual/baseline/longest_theorems_proved_old.txt"),
        **{f"seed{s+1}": dfs_seed_sets[s]["rm"] for s in range(3)},
    },
}

In [3]:
from itertools import combinations

def print_set_info(dfs):
    names = {k: set(df['theorem_name']) for k, df in dfs.items()}

    for k, s in names.items():
        print(f"{k}: {len(s)} theorems")

    print("\nJaccard Similarity:")
    for a, b in combinations(names, 2):
        intersection = len(names[a] & names[b])
        union = len(names[a] | names[b])
        print(f"  {a} x {b}: {((intersection / union) * 100):.2f}% ({intersection}/{union})")

    common_all = set.intersection(*names.values())
    print(f"\nCommon to all {len(names)}: {len(common_all)}")

    union_all = set.union(*names.values())
    print(f"Union: {len(union_all)}")

In [4]:
for i, dfs in enumerate(dfs_seed_sets):
    print(f"SEED: {i}")
    print("---------------------")
    print_set_info(dfs)
    print("=====================")

SEED: 0
---------------------
nb: 171 theorems
nm: 170 theorems
rb: 187 theorems
rm: 158 theorems

Jaccard Similarity:
  nb x nm: 57.14% (124/217)
  nb x rb: 57.71% (131/227)
  nb x rm: 54.46% (116/213)
  nm x rb: 55.90% (128/229)
  nm x rm: 55.45% (117/211)
  rb x rm: 52.65% (119/226)

Common to all 4: 84
Union: 262
SEED: 1
---------------------
nb: 185 theorems
nm: 172 theorems
rb: 187 theorems
rm: 160 theorems

Jaccard Similarity:
  nb x nm: 53.22% (124/233)
  nb x rb: 58.97% (138/234)
  nb x rm: 51.98% (118/227)
  nm x rb: 56.77% (130/229)
  nm x rm: 53.70% (116/216)
  rb x rm: 50.22% (116/231)

Common to all 4: 82
Union: 272
SEED: 2
---------------------
nb: 184 theorems
nm: 169 theorems
rb: 178 theorems
rm: 159 theorems

Jaccard Similarity:
  nb x nm: 59.01% (131/222)
  nb x rb: 60.18% (136/226)
  nb x rm: 53.81% (120/223)
  nm x rb: 54.91% (123/224)
  nm x rm: 54.72% (116/212)
  rb x rm: 54.59% (119/218)

Common to all 4: 88
Union: 264


In [5]:
for seed, model in enumerate(dfs_models_sets.keys()):
    print(f"MODEL: {model}")
    print("---------------------")
    print_set_info(dfs_models_sets[model])
    print("=====================")

MODEL: Normal Baseline
---------------------
prev0: 128 theorems
seed1: 171 theorems
seed2: 185 theorems
seed3: 184 theorems

Jaccard Similarity:
  prev0 x seed1: 20.56% (51/248)
  prev0 x seed2: 20.38% (53/260)
  prev0 x seed3: 21.40% (55/257)
  seed1 x seed2: 71.15% (148/208)
  seed1 x seed3: 72.33% (149/206)
  seed2 x seed3: 74.88% (158/211)

Common to all 4: 41
Union: 287
MODEL: Normal Modular
---------------------
prev0: 129 theorems
seed1: 170 theorems
seed2: 172 theorems
seed3: 169 theorems

Jaccard Similarity:
  prev0 x seed1: 20.56% (51/248)
  prev0 x seed2: 20.88% (52/249)
  prev0 x seed3: 20.16% (50/248)
  seed1 x seed2: 95.43% (167/175)
  seed1 x seed3: 92.61% (163/176)
  seed2 x seed3: 93.75% (165/176)

Common to all 4: 49
Union: 254
MODEL: Residual Baseline
---------------------
prev0: 127 theorems
seed1: 187 theorems
seed2: 187 theorems
seed3: 178 theorems

Jaccard Similarity:
  prev0 x seed1: 17.16% (46/268)
  prev0 x seed2: 18.94% (50/264)
  prev0 x seed3: 18.22% (47/2

In [6]:
# from venny4py.venny4py import venny4py

# venny4py(sets=names, out='.', ext='png', dpi=150)

In [7]:
# all_names = sorted(union_all)
# membership = pd.DataFrame(
#     {k: [n in names[k] for n in all_names] for k in names},
#     index=all_names
# )
# print(membership)

In [8]:
# collapsed = {
#     k: df.groupby('theorem_name', as_index=False)['expansions'].first().set_index('theorem_name')
#     for k, df in dfs.items()
# }

# expansions = pd.concat(
#     {k: c['expansions'] for k, c in collapsed.items()},
#     axis=1
# )
# expansions_common = expansions.loc[list(common_all)]
# print(expansions_common)

In [9]:
# winners = expansions_common.idxmin(axis=1)
# # print(winners)

# print(winners.value_counts())